In [1]:
import pandas as pd
from silver_staging_utils import connect_to_postgres, read_table
import numpy as np

In [2]:
conn = connect_to_postgres()
if conn:
    df = read_table("SELECT * FROM silver.categories LIMIT 100", conn)
    conn.close()
    

✅ Conectado a PostgreSQL


/workspaces/tesis-ivan-gennaro/scripts/silver_staging/silver_staging_utils.py:44: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


In [3]:
df.head()

,snapshot_date,supermarket,category_lvl1_name,category_lvl2_name,category_lvl3_name,category_lvl1_id,category_lvl2_id,category_lvl3_id,category_lvl1_slug,category_lvl2_slug,category_lvl3_slug,created_at
0,2025-08-10,biggie,Alimentos Especiales,None,None,6,None,None,alimentos-especiales,None,None,2025-10-27 00:10:31.919386
1,2025-08-10,biggie,Almacén,None,None,1,None,None,almacen,None,None,2025-10-27 00:10:31.919386
2,2025-08-10,biggie,Asado,None,None,246,None,None,asado,None,None,2025-10-27 00:10:31.919386
3,2025-08-10,biggie,Bebes,None,None,41,None,None,bebes,None,None,2025-10-27 00:10:31.919386
4,2025-08-10,biggie,Bebidas con Alcohol,None,None,3,None,None,bebidas-con-alcohol,None,None,2025-10-27 00:10:31.919386


### Products

In [4]:
query = '''
SELECT
    *
FROM silver.products
WHERE
    snapshot_date IN (
        SELECT MAX(snapshot_date) FROM silver.products  
    )
'''
print(query)


SELECT
    *
FROM silver.products
WHERE
    snapshot_date IN (
        SELECT MAX(snapshot_date) FROM silver.products  
    )



In [5]:
conn = connect_to_postgres()
if conn:
    df = read_table(query, conn)
    conn.close()

✅ Conectado a PostgreSQL


/workspaces/tesis-ivan-gennaro/scripts/silver_staging/silver_staging_utils.py:44: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45260 entries, 0 to 45259
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   snapshot_date    45260 non-null  object        
 1   supermarket      45260 non-null  object        
 2   product_id       16880 non-null  object        
 3   product_name     45260 non-null  object        
 4   brand            15990 non-null  object        
 5   price            45260 non-null  object        
 6   unit_of_measure  23476 non-null  object        
 7   is_on_promotion  6770 non-null   object        
 8   promotion_price  9045 non-null   object        
 9   category_slug    45260 non-null  object        
 10  ingestion_time   45260 non-null  datetime64[ns]
 11  created_at       45260 non-null  datetime64[ns]
dtypes: datetime64[ns](2), object(10)
memory usage: 4.1+ MB


In [7]:
df['snapshot_date'].unique()

array([datetime.date(2025, 10, 26)], dtype=object)

In [8]:
df['supermarket'].unique()

array(['biggie', 'real', 'stock', 'casarica'], dtype=object)

#### Final Price creation

Price problem with Casa Rica

In [9]:
df["price"] = (
    df["price"]
    .astype(str)
    .str.replace(r"[^\d,\.]", "", regex=True)
    .str.replace(".", "", regex=False) 
    .str.replace(",", ".", regex=False)
    .pipe(pd.to_numeric, errors="coerce")
)

Coalesce to build final price

In [10]:
df.loc[df['promotion_price'] == '0', 'promotion_price'] = None

In [11]:
df["final_price"] = (
    df["promotion_price"]
        .combine_first(df['price'])
)

Validations

In [12]:
df.loc[df['supermarket'] == 'casarica', 'price'].head()

33586    120000
33587     32500
33588     23000
33589    115000
33590     59950
Name: price, dtype: int64

In [13]:
df['promotion_price'].unique()

array([None, '36900', '12500', ..., '42000.0', '17050.0', '9100.0'],
      shape=(1372,), dtype=object)

In [14]:
# Casa Rica get numeric
df.loc[df['supermarket'] == 'casarica', ['price', 'promotion_price', 'final_price']]

,price,promotion_price,final_price
33586,120000,None,120000
33587,32500,None,32500
33588,23000,None,23000
33589,115000,None,115000
33590,59950,None,59950
...,...,...,...
45255,27600,None,27600
45256,8700,None,8700
45257,46250,None,46250
45258,46250,None,46250


In [15]:
# Coalesce
df.loc[(df['supermarket'] == 'biggie') & (df['promotion_price'].notna()), ['price', 'promotion_price', 'final_price']]

,price,promotion_price,final_price
148,40900,36900,36900
152,14850,12500,12500
154,26900,22900,22900
155,13900,11500,11500
157,23900,20350,20350
...,...,...,...
6757,3500,3150,3150
6759,15500,14250,14250
6760,20000,16000,16000
6761,20000,16000,16000


In [17]:
df.loc[(df['supermarket'] == 'real') & (df['promotion_price'].notna()), ['price', 'promotion_price', 'final_price']]

,price,promotion_price,final_price
6778,5000,4500.0,4500.0
6779,5000,4500.0,4500.0
6780,2650,2400.0,2400.0
6781,2650,2400.0,2400.0
6782,18950,17000.0,17000.0
...,...,...,...
16806,19050,17050.0,17050.0
16826,55000,43500.0,43500.0
16839,10200,9100.0,9100.0
16840,21550,19200.0,19200.0


In [18]:
df.head()

,snapshot_date,supermarket,product_id,product_name,brand,price,unit_of_measure,is_on_promotion,promotion_price,category_slug,ingestion_time,created_at,final_price
0,2025-10-26,biggie,7891173022868,Resma A4 Chamex 500 Hojas.,None,46600,Unidades,False,None,libreria,2025-10-26 10:15:07.312096,2025-10-27 00:10:40.570539,46600
1,2025-10-26,biggie,77960863,Pegamento Voligoma de 30 ml.,None,11000,Unidades,False,None,libreria,2025-10-26 10:15:07.312096,2025-10-27 00:10:40.570539,11000
2,2025-10-26,biggie,70330129665,Boligrafo Bic cristal color negro 1 unidad.,None,1500,Unidades,False,None,libreria,2025-10-26 10:15:07.312096,2025-10-27 00:10:40.570539,1500
3,2025-10-26,biggie,70330129627,Boligrafo Bic cristal color azul 1 unidad.,None,1500,Unidades,False,None,libreria,2025-10-26 10:15:07.312096,2025-10-27 00:10:40.570539,1500
4,2025-10-26,biggie,7806505048317,Silicona Liquida Torre de 30 ml.,None,4850,Unidades,False,None,libreria,2025-10-26 10:15:07.312096,2025-10-27 00:10:40.570539,4850
